In [1]:
# ============================================================
# Validation Step 2 — Stratified Manual-Audit Sampling
# V18.1 schema-safe sampler
#
# Reads from:
#   C:\Android Mobile App\ICST2026_Ext\MainDataset.csv
#   C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv
#   or auto-extracts from:
#   C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.zip
#
# Writes to:
#   C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample
#
# Main goals
# - Build a stratified run×style sample for Step 2 manual review
# - Ensure coverage across:
#     * style
#     * Base / non-Base
#     * Layer 2 observable / not observable
#     * single-style / multi-style run
#     * step-telemetry available / not available
# - Also collect targeted step-level records for manual review of:
#     * step activity group
#     * execution role
#     * overhead phase
#
# Current-study adjustments
# - Base subset is now defined consistently with the paper:
#     run_attempt == 1 AND usable verdict == True
# - Instrumentation verdict/conclusion is not part of the study-facing
#   Step 2 manual checklist
# - Adds run_link for each sampled main row and audit-sheet row
# - Adds targeted coverage for newer edge-case structures already
#   represented in MainDataset:
#     * repeated same-style runs
#     * matrix-expanded same-style runs
#     * cross-job execution-window cases
#     * parallel same-style cases
#
# IMPORTANT
# - Random seed remains unchanged
# - Core sampling design remains unchanged
#
# Outputs
# - step2_sample_main_records.csv
# - step2_sample_step_records.csv
# - step2_sample_main_with_step_examples.csv
# - step2_sample_summary.txt
# - step2_sample_strata_counts_main.csv
# - step2_sample_strata_counts_steps.csv
# - step2_sample_design_audit_sheet.csv
# ============================================================

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
OUT_DIR = BASE_DIR / r"0.2-Validation\Step-2-Stratified_Sample"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_PATH = BASE_DIR / "MainDataset.csv"
STEPS_CSV_PATH = BASE_DIR / "run_steps_v16_stage3_breakdown.csv"
STEPS_ZIP_PATH = BASE_DIR / "run_steps_v16_stage3_breakdown.zip"

# ------------------------------------------------------------
# Sampling knobs
# ------------------------------------------------------------
RANDOM_SEED = 20260317

# Core balanced sample: per style from Base subset
CORE_PER_STYLE = 15

# Extra targeted pools (main-record level)
EXTRA_LAYER2_OBS = 20
EXTRA_MULTISTYLE = 15
EXTRA_NONBASE = 15
EXTRA_EDGE_OR_AMBIG = 10
EXTRA_COMPLEX_SAME_STYLE = 12   # repeated/matrix/cross-job/parallel cases

# Step-level targeted sampling for category-dimension validation
STEP_PER_ACTIVITY_GROUP = 4
STEP_PER_EXEC_ROLE = 4
STEP_PER_OVERHEAD_PHASE = 4

# Cap duplicate expansion in step sample
MAX_STEPS_PER_MAIN_RECORD = 3

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def safe_series(df, col, dtype=None, fill_value=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series(fill_value, index=df.index, dtype=dtype)

def first_present_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def get_first_present_series(df, candidates, dtype=None, fill_value=np.nan):
    c = first_present_col(df, candidates)
    if c is not None:
        return df[c], c
    return pd.Series(fill_value, index=df.index, dtype=dtype), None

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def to_bool_loose(s):
    x = s.astype(str).str.strip().str.lower()
    true_set = {"1", "true", "yes", "y", "complete", "completed", "ok"}
    false_set = {"0", "false", "no", "n", "incomplete", "missing", "failed"}
    out = pd.Series(np.nan, index=s.index, dtype="object")
    out.loc[x.isin(true_set)] = True
    out.loc[x.isin(false_set)] = False
    out.loc[s.eq(True)] = True
    out.loc[s.eq(False)] = False
    return out

def bool_like_yes(s):
    x = s.astype(str).str.strip().str.lower()
    return x.isin({"1", "true", "yes", "y", "ok"})

def canon_style_token(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    mapping = {
        "community": "Community",
        "custom": "Custom",
        "third-party": "Third-Party",
        "third party": "Third-Party",
        "gmd": "GMD",
        "real-device": "Real-Device",
        "real device": "Real-Device",
    }
    return mapping.get(s.lower(), s)

def sample_n(df, n, seed=RANDOM_SEED):
    if len(df) == 0:
        return df.copy()
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed)

def sample_per_group(df, group_col, n_per_group, seed=RANDOM_SEED):
    out = []
    if group_col not in df.columns:
        return df.iloc[0:0].copy()
    for i, (_, part) in enumerate(df.groupby(group_col, dropna=False)):
        out.append(sample_n(part, n_per_group, seed + i))
    if not out:
        return df.iloc[0:0].copy()
    return pd.concat(out, ignore_index=True)

def sample_without_duplicates(candidates, already_keys, key_cols, n, seed=RANDOM_SEED):
    if len(candidates) == 0 or n <= 0:
        return candidates.iloc[0:0].copy()

    cand = candidates.copy()
    if already_keys:
        key_df = pd.DataFrame(list(already_keys), columns=key_cols)
        cand = cand.merge(
            key_df.assign(_already=1),
            on=key_cols,
            how="left"
        )
        cand = cand[cand["_already"].isna()].drop(columns="_already")

    if len(cand) == 0:
        return cand.iloc[0:0].copy()

    return sample_n(cand, n, seed=seed).copy()

def add_source_tag(df, tag):
    out = df.copy()
    out["sample_source"] = tag
    return out

def build_run_link(full_name_series, run_id_series):
    full_name_clean = full_name_series.fillna("").astype(str).str.strip()
    run_id_clean = run_id_series.fillna("").astype(str).str.strip()
    return np.where(
        (full_name_clean != "") & (run_id_clean != ""),
        "https://github.com/" + full_name_clean + "/actions/runs/" + run_id_clean,
        ""
    )

# ------------------------------------------------------------
# Read inputs
# ------------------------------------------------------------
print("Reading MainDataset...")
main_df = pd.read_csv(MAIN_PATH, low_memory=False)
print("MainDataset shape:", main_df.shape)

if STEPS_CSV_PATH.exists():
    steps_path_to_use = STEPS_CSV_PATH
elif STEPS_ZIP_PATH.exists():
    print("Extracting step breakdown CSV from zip...")
    with zipfile.ZipFile(STEPS_ZIP_PATH, "r") as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise FileNotFoundError("No CSV found inside run_steps_v16_stage3_breakdown.zip")
        first_csv = csv_names[0]
        zf.extract(first_csv, OUT_DIR)
        extracted = OUT_DIR / first_csv
        if extracted.name != STEPS_CSV_PATH.name:
            target = OUT_DIR / STEPS_CSV_PATH.name
            extracted.replace(target)
            steps_path_to_use = target
        else:
            steps_path_to_use = extracted
else:
    raise FileNotFoundError(
        f"Could not find either:\n- {STEPS_CSV_PATH}\n- {STEPS_ZIP_PATH}"
    )

print("Reading step breakdown...")
steps_df = pd.read_csv(steps_path_to_use, low_memory=False)
print("Step breakdown shape:", steps_df.shape)

# ------------------------------------------------------------
# Basic normalization
# ------------------------------------------------------------
if "style" in main_df.columns:
    main_df["style"] = main_df["style"].map(canon_style_token)

if "target_style" in steps_df.columns:
    steps_df["target_style"] = steps_df["target_style"].map(canon_style_token)

main_df["run_id"] = safe_series(main_df, "run_id", dtype="object", fill_value="").astype(str)
steps_df["run_id"] = safe_series(steps_df, "run_id", dtype="object", fill_value="").astype(str)

main_df["full_name"] = safe_series(main_df, "full_name", dtype="object", fill_value="").astype(str)
steps_df["full_name"] = safe_series(steps_df, "full_name", dtype="object", fill_value="").astype(str)

main_df["style"] = safe_series(main_df, "style", dtype="object", fill_value=np.nan)
steps_df["target_style"] = safe_series(steps_df, "target_style", dtype="object", fill_value=np.nan)

# ------------------------------------------------------------
# Resolve controller fields
# Base subset = attempt == 1 AND usable verdict == True
# ------------------------------------------------------------
attempt_series, attempt_col = get_first_present_series(
    main_df,
    ["attempt", "run_attempt", "study_attempt"],
    dtype="object",
    fill_value=np.nan
)

usable_verdict_series, usable_verdict_col = get_first_present_series(
    main_df,
    [
        "controller_usable_verdict",
        "usable_verdict",
        "study_controller_usable_verdict",
        "controller_run_usable_verdict",
    ],
    dtype="object",
    fill_value=np.nan
)

run_ctrl_series, run_ctrl_col = get_first_present_series(
    main_df,
    [
        "controller_run_verdict_complete",
        "run_verdict_complete",
        "study_controller_run_verdict_complete",
        "controller_run_complete",
    ],
    dtype="object",
    fill_value=np.nan
)

inst_ctrl_series, inst_ctrl_col = get_first_present_series(
    main_df,
    [
        "controller_instrumentation_verdict_complete",
        "instrumentation_verdict_complete",
        "study_controller_instrumentation_verdict_complete",
        "controller_instrumentation_complete",
    ],
    dtype="object",
    fill_value=np.nan
)

main_df["attempt_resolved"] = to_num(attempt_series)
main_df["controller_usable_verdict_resolved"] = to_bool_loose(usable_verdict_series)
main_df["controller_run_verdict_complete_resolved"] = to_bool_loose(run_ctrl_series)
main_df["controller_instrumentation_verdict_complete_resolved"] = to_bool_loose(inst_ctrl_series)

main_df["base_subset_flag"] = np.where(
    main_df["attempt_resolved"].eq(1) &
    main_df["controller_usable_verdict_resolved"].eq(True),
    "Base",
    "Non-Base"
)

# ------------------------------------------------------------
# Layer 2 observability
# ------------------------------------------------------------
l2_candidates = [
    "study_pre_invocation_selected_stage3_seconds",
    "study_invocation_execution_window_selected_stage3_seconds",
    "study_post_invocation_selected_stage3_seconds",
]
present_l2_cols = [c for c in l2_candidates if c in main_df.columns]

if len(present_l2_cols) == 3:
    main_df["layer2_observable"] = np.where(
        main_df[l2_candidates].notna().all(axis=1),
        "Yes", "No"
    )
else:
    l2_anchor_candidates = [
        "study_matched_invocation_step_started_at",
        "study_invocation_execution_end_step_completed_at",
        "study_run_boundary_start_at",
        "study_run_boundary_end_at",
    ]
    present_anchor_cols = [c for c in l2_anchor_candidates if c in main_df.columns]
    if len(present_anchor_cols) >= 4:
        main_df["layer2_observable"] = np.where(
            main_df[l2_anchor_candidates].notna().all(axis=1),
            "Yes", "No"
        )
    else:
        main_df["layer2_observable"] = "Unknown"

# ------------------------------------------------------------
# Multi-style vs single-style run
# ------------------------------------------------------------
style_counts = (
    main_df[["full_name", "run_id", "style"]]
    .drop_duplicates()
    .groupby(["full_name", "run_id"], dropna=False)
    .size()
    .rename("style_count_in_run")
    .reset_index()
)

main_df = main_df.merge(style_counts, on=["full_name", "run_id"], how="left")
main_df["style_multiplicity"] = np.where(
    main_df["style_count_in_run"].fillna(0).ge(2),
    "Multi-Style",
    "Single-Style"
)

# ------------------------------------------------------------
# Step telemetry availability at run×style level
# ------------------------------------------------------------
step_presence = (
    steps_df[["full_name", "run_id", "target_style"]]
    .drop_duplicates()
    .assign(step_telemetry_available="Yes")
)

main_df = main_df.merge(
    step_presence,
    left_on=["full_name", "run_id", "style"],
    right_on=["full_name", "run_id", "target_style"],
    how="left"
)
main_df["step_telemetry_available"] = main_df["step_telemetry_available"].fillna("No")
main_df = main_df.drop(columns=[c for c in ["target_style"] if c in main_df.columns])

# ------------------------------------------------------------
# Newer structure indicators already present in MainDataset
# ------------------------------------------------------------
main_df["same_style_repeated_flag"] = bool_like_yes(
    safe_series(main_df, "study_style_repeated_same_style_flag", dtype="object", fill_value="false")
)
main_df["matrix_expanded_flag"] = bool_like_yes(
    safe_series(main_df, "study_style_matrix_expanded_flag", dtype="object", fill_value="false")
)
main_df["cross_job_execution_window_flag"] = bool_like_yes(
    safe_series(main_df, "study_cross_job_execution_window_flag", dtype="object", fill_value="false")
)
main_df["parallel_same_style_flag"] = bool_like_yes(
    safe_series(main_df, "study_style_parallel_same_style_flag", dtype="object", fill_value="false")
)

main_df["same_style_complexity_class"] = safe_series(
    main_df, "study_style_same_style_complexity_class", dtype="object", fill_value=np.nan
)

# ------------------------------------------------------------
# Edge-case tagging
# ------------------------------------------------------------
main_df["edge_case_flag"] = "No"

main_df.loc[
    (main_df["base_subset_flag"] == "Non-Base") &
    (main_df["layer2_observable"] == "Yes"),
    "edge_case_flag"
] = "Yes"

main_df.loc[
    (main_df["style_multiplicity"] == "Multi-Style") &
    (main_df["layer2_observable"] == "Yes"),
    "edge_case_flag"
] = "Yes"

main_df.loc[
    main_df["same_style_repeated_flag"] |
    main_df["matrix_expanded_flag"] |
    main_df["cross_job_execution_window_flag"] |
    main_df["parallel_same_style_flag"],
    "edge_case_flag"
] = "Yes"

# finer label for reporting
main_df["complex_same_style_flag"] = np.where(
    main_df["same_style_repeated_flag"] |
    main_df["matrix_expanded_flag"] |
    main_df["cross_job_execution_window_flag"] |
    main_df["parallel_same_style_flag"],
    "Yes",
    "No"
)

# ------------------------------------------------------------
# Build main-record sample
# ------------------------------------------------------------
key_cols = ["full_name", "run_id", "style"]

sample_parts = []
already_keys = set()

base_core_pool = main_df[
    (main_df["base_subset_flag"] == "Base") &
    (main_df["style"].isin(["Community", "Custom", "Third-Party", "GMD"]))
].copy()

core = sample_per_group(base_core_pool, "style", CORE_PER_STYLE, seed=RANDOM_SEED)
core = add_source_tag(core, "core_base_by_style")
sample_parts.append(core)
for _, r in core[key_cols].drop_duplicates().iterrows():
    already_keys.add(tuple(r[c] for c in key_cols))

layer2_pool = main_df[main_df["layer2_observable"] == "Yes"].copy()
extra_l2 = sample_without_duplicates(layer2_pool, already_keys, key_cols, EXTRA_LAYER2_OBS, seed=RANDOM_SEED + 100)
extra_l2 = add_source_tag(extra_l2, "extra_layer2_observable")
sample_parts.append(extra_l2)
for _, r in extra_l2[key_cols].drop_duplicates().iterrows():
    already_keys.add(tuple(r[c] for c in key_cols))

multistyle_pool = main_df[main_df["style_multiplicity"] == "Multi-Style"].copy()
extra_multi = sample_without_duplicates(multistyle_pool, already_keys, key_cols, EXTRA_MULTISTYLE, seed=RANDOM_SEED + 200)
extra_multi = add_source_tag(extra_multi, "extra_multistyle")
sample_parts.append(extra_multi)
for _, r in extra_multi[key_cols].drop_duplicates().iterrows():
    already_keys.add(tuple(r[c] for c in key_cols))

nonbase_pool = main_df[main_df["base_subset_flag"] == "Non-Base"].copy()
extra_nonbase = sample_without_duplicates(nonbase_pool, already_keys, key_cols, EXTRA_NONBASE, seed=RANDOM_SEED + 300)
extra_nonbase = add_source_tag(extra_nonbase, "extra_nonbase")
sample_parts.append(extra_nonbase)
for _, r in extra_nonbase[key_cols].drop_duplicates().iterrows():
    already_keys.add(tuple(r[c] for c in key_cols))

edge_pool = main_df[main_df["edge_case_flag"] == "Yes"].copy()
extra_edge = sample_without_duplicates(edge_pool, already_keys, key_cols, EXTRA_EDGE_OR_AMBIG, seed=RANDOM_SEED + 400)
extra_edge = add_source_tag(extra_edge, "extra_edge_cases")
sample_parts.append(extra_edge)
for _, r in extra_edge[key_cols].drop_duplicates().iterrows():
    already_keys.add(tuple(r[c] for c in key_cols))

complex_same_style_pool = main_df[main_df["complex_same_style_flag"] == "Yes"].copy()
extra_complex = sample_without_duplicates(
    complex_same_style_pool,
    already_keys,
    key_cols,
    EXTRA_COMPLEX_SAME_STYLE,
    seed=RANDOM_SEED + 450
)
extra_complex = add_source_tag(extra_complex, "extra_complex_same_style")
sample_parts.append(extra_complex)
for _, r in extra_complex[key_cols].drop_duplicates().iterrows():
    already_keys.add(tuple(r[c] for c in key_cols))

main_sample = pd.concat(sample_parts, ignore_index=True) if sample_parts else main_df.iloc[0:0].copy()
main_sample = (
    main_sample
    .sort_values(key_cols + ["sample_source"])
    .drop_duplicates(subset=key_cols, keep="first")
    .reset_index(drop=True)
)
main_sample["sample_record_id"] = ["M%04d" % i for i in range(1, len(main_sample) + 1)]
main_sample["run_link"] = build_run_link(main_sample["full_name"], main_sample["run_id"])

# ------------------------------------------------------------
# Step-level sample for grouping/category-dimension review
# ------------------------------------------------------------
activity_col = first_present_col(
    steps_df,
    ["step_activity_group", "study_step_activity_group", "activity_group", "step_group"]
)
exec_role_col = first_present_col(
    steps_df,
    ["execution_role", "study_execution_role", "step_execution_role"]
)
overhead_col = first_present_col(
    steps_df,
    ["overhead_phase", "study_overhead_phase", "step_overhead_phase"]
)

steps_df["step_activity_group_resolved"] = safe_series(steps_df, activity_col, dtype="object", fill_value=np.nan) if activity_col else np.nan
steps_df["execution_role_resolved"] = safe_series(steps_df, exec_role_col, dtype="object", fill_value=np.nan) if exec_role_col else np.nan
steps_df["overhead_phase_resolved"] = safe_series(steps_df, overhead_col, dtype="object", fill_value=np.nan) if overhead_col else np.nan

step_main_keys = main_sample[key_cols].drop_duplicates().copy()
step_pool = steps_df.merge(
    step_main_keys,
    left_on=["full_name", "run_id", "target_style"],
    right_on=["full_name", "run_id", "style"],
    how="inner"
)

if len(step_pool) == 0:
    step_pool = steps_df.merge(
        main_sample[["full_name", "run_id"]].drop_duplicates(),
        on=["full_name", "run_id"],
        how="inner"
    )

step_sample_parts = []

if "step_activity_group_resolved" in step_pool.columns:
    part = step_pool[step_pool["step_activity_group_resolved"].notna()].copy()
    part = sample_per_group(part, "step_activity_group_resolved", STEP_PER_ACTIVITY_GROUP, seed=RANDOM_SEED + 500)
    part = add_source_tag(part, "step_activity_group_strata")
    step_sample_parts.append(part)

if "execution_role_resolved" in step_pool.columns:
    part = step_pool[step_pool["execution_role_resolved"].notna()].copy()
    part = sample_per_group(part, "execution_role_resolved", STEP_PER_EXEC_ROLE, seed=RANDOM_SEED + 600)
    part = add_source_tag(part, "step_execution_role_strata")
    step_sample_parts.append(part)

if "overhead_phase_resolved" in step_pool.columns:
    part = step_pool[step_pool["overhead_phase_resolved"].notna()].copy()
    part = sample_per_group(part, "overhead_phase_resolved", STEP_PER_OVERHEAD_PHASE, seed=RANDOM_SEED + 700)
    part = add_source_tag(part, "step_overhead_phase_strata")
    step_sample_parts.append(part)

step_sample = pd.concat(step_sample_parts, ignore_index=True) if step_sample_parts else step_pool.iloc[0:0].copy()

step_dedup_keys = [c for c in ["full_name", "run_id", "target_style", "job_name", "step_name", "started_at", "completed_at"] if c in step_sample.columns]
if len(step_dedup_keys) == 0:
    step_dedup_keys = [c for c in ["full_name", "run_id", "job_name", "step_name"] if c in step_sample.columns]

if len(step_sample):
    step_sample = (
        step_sample
        .sort_values(step_dedup_keys + ["sample_source"] if "sample_source" in step_sample.columns else step_dedup_keys)
        .drop_duplicates(subset=step_dedup_keys, keep="first")
        .reset_index(drop=True)
    )

step_sample["join_style_for_main"] = safe_series(step_sample, "target_style", dtype="object", fill_value=np.nan)
step_sample = step_sample.merge(
    main_sample[["sample_record_id", "full_name", "run_id", "style"]],
    left_on=["full_name", "run_id", "join_style_for_main"],
    right_on=["full_name", "run_id", "style"],
    how="left",
    suffixes=("", "_main")
)

if "sample_record_id" in step_sample.columns:
    step_sample["_rank_in_main"] = step_sample.groupby("sample_record_id").cumcount() + 1
    step_sample = step_sample[
        step_sample["_rank_in_main"].le(MAX_STEPS_PER_MAIN_RECORD) | step_sample["sample_record_id"].isna()
    ].copy()
    step_sample = step_sample.drop(columns=["_rank_in_main"])

step_sample = step_sample.reset_index(drop=True)
step_sample["sample_step_id"] = ["S%04d" % i for i in range(1, len(step_sample) + 1)]

# ------------------------------------------------------------
# Main sample + linked step examples
# ------------------------------------------------------------
step_examples = (
    step_sample
    .sort_values(["sample_record_id", "sample_step_id"])
    .groupby("sample_record_id", dropna=False)
    .head(1)
    .copy()
)

main_with_step_examples = main_sample.merge(
    step_examples[
        [c for c in [
            "sample_record_id",
            "sample_step_id",
            "job_name",
            "step_name",
            "step_activity_group_resolved",
            "execution_role_resolved",
            "overhead_phase_resolved",
            "sample_source",
        ] if c in step_examples.columns]
    ].rename(columns={"sample_source": "linked_step_sample_source"}),
    on="sample_record_id",
    how="left"
)

# ------------------------------------------------------------
# Build strata count tables
# ------------------------------------------------------------
main_group_cols = [
    c for c in [
        "sample_source",
        "style",
        "base_subset_flag",
        "layer2_observable",
        "style_multiplicity",
        "step_telemetry_available",
        "complex_same_style_flag",
        "same_style_complexity_class",
    ] if c in main_sample.columns
]

main_strata_counts = (
    main_sample.groupby(main_group_cols, dropna=False)
    .size()
    .rename("n")
    .reset_index()
)

step_group_cols = [c for c in ["sample_source", "target_style", "step_activity_group_resolved", "execution_role_resolved", "overhead_phase_resolved"] if c in step_sample.columns]
if step_group_cols:
    step_strata_counts = (
        step_sample.groupby(step_group_cols, dropna=False)
        .size()
        .rename("n")
        .reset_index()
    )
else:
    step_strata_counts = pd.DataFrame()

# ------------------------------------------------------------
# Audit sheet
# ------------------------------------------------------------
audit_cols = [
    "sample_record_id",
    "sample_source",
    "full_name",
    "workflow_identifier",
    "workflow_id",
    "run_id",
    "run_link",
    "style",
    "base_subset_flag",
    "layer2_observable",
    "style_multiplicity",
    "step_telemetry_available",
    "edge_case_flag",
    "attempt_resolved",
    "controller_usable_verdict_resolved",
    "controller_run_verdict_complete_resolved",

    # added reviewer context fields
    "complex_same_style_flag",
    "same_style_repeated_flag",
    "matrix_expanded_flag",
    "cross_job_execution_window_flag",
    "parallel_same_style_flag",
    "same_style_complexity_class",
]
audit_cols = [c for c in audit_cols if c in main_with_step_examples.columns]

audit_sheet = main_with_step_examples[audit_cols].copy()

review_cols = [
    "audit_in_scope",
    "audit_instrumentation_executed",
    "audit_style_correct",
    "audit_layer1_boundaries_correct",
    "audit_layer2_invocation_anchor_correct",
    "audit_layer2_execution_end_anchor_correct",
    "audit_step_activity_group_correct",
    "audit_execution_role_correct",
    "audit_overhead_phase_correct",
    "audit_attempt_correct",
    "audit_usable_verdict_correct",
    "audit_notes",
]
for c in review_cols:
    audit_sheet[c] = ""

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
main_sample.to_csv(OUT_DIR / "step2_sample_main_records.csv", index=False)
step_sample.to_csv(OUT_DIR / "step2_sample_step_records.csv", index=False)
main_with_step_examples.to_csv(OUT_DIR / "step2_sample_main_with_step_examples.csv", index=False)
main_strata_counts.to_csv(OUT_DIR / "step2_sample_strata_counts_main.csv", index=False)
step_strata_counts.to_csv(OUT_DIR / "step2_sample_strata_counts_steps.csv", index=False)
audit_sheet.to_csv(OUT_DIR / "step2_sample_design_audit_sheet.csv", index=False)

# ------------------------------------------------------------
# Save summary note
# ------------------------------------------------------------
summary_lines = []
summary_lines.append("Validation Step 2 — Stratified Manual-Audit Sample")
summary_lines.append("=" * 60)
summary_lines.append(f"Random seed: {RANDOM_SEED}")
summary_lines.append("")
summary_lines.append("Resolved controller columns:")
summary_lines.append(f" - attempt: {attempt_col}")
summary_lines.append(f" - usable verdict: {usable_verdict_col}")
summary_lines.append(f" - run verdict complete (context only): {run_ctrl_col}")
summary_lines.append(f" - instrumentation verdict complete (context only, not in checklist): {inst_ctrl_col}")
summary_lines.append("")
summary_lines.append("Resolved step category columns:")
summary_lines.append(f" - step activity group: {activity_col}")
summary_lines.append(f" - execution role: {exec_role_col}")
summary_lines.append(f" - overhead phase: {overhead_col}")
summary_lines.append("")
summary_lines.append("Run link column added: Yes")
summary_lines.append("Complex same-style targeted pool added: Yes")
summary_lines.append("")
summary_lines.append(f"Main sample size: {len(main_sample)}")
summary_lines.append(f"Step sample size: {len(step_sample)}")
summary_lines.append("")
summary_lines.append("Main sample by source:")
if "sample_source" in main_sample.columns:
    summary_lines.append(main_sample["sample_source"].value_counts(dropna=False).to_string())
summary_lines.append("")
summary_lines.append("Main sample by style:")
if "style" in main_sample.columns:
    summary_lines.append(main_sample["style"].value_counts(dropna=False).to_string())
summary_lines.append("")
summary_lines.append("Main sample by Base / Non-Base:")
summary_lines.append(main_sample["base_subset_flag"].value_counts(dropna=False).to_string())
summary_lines.append("")
summary_lines.append("Main sample by Layer 2 observability:")
summary_lines.append(main_sample["layer2_observable"].value_counts(dropna=False).to_string())
summary_lines.append("")
summary_lines.append("Main sample by style multiplicity:")
summary_lines.append(main_sample["style_multiplicity"].value_counts(dropna=False).to_string())
summary_lines.append("")
if "complex_same_style_flag" in main_sample.columns:
    summary_lines.append("Main sample by complex same-style flag:")
    summary_lines.append(main_sample["complex_same_style_flag"].value_counts(dropna=False).to_string())
    summary_lines.append("")
if "same_style_complexity_class" in main_sample.columns:
    summary_lines.append("Main sample by same-style complexity class:")
    summary_lines.append(main_sample["same_style_complexity_class"].value_counts(dropna=False).to_string())
    summary_lines.append("")
summary_lines.append("Step sample by category dimensions:")
for c in ["step_activity_group_resolved", "execution_role_resolved", "overhead_phase_resolved"]:
    if c in step_sample.columns:
        summary_lines.append(f"\n{c}:")
        summary_lines.append(step_sample[c].value_counts(dropna=False).to_string())

(OUT_DIR / "step2_sample_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")

# ------------------------------------------------------------
# Console output
# ------------------------------------------------------------
print("\nSaved outputs to:", OUT_DIR)
print(" - step2_sample_main_records.csv")
print(" - step2_sample_step_records.csv")
print(" - step2_sample_main_with_step_examples.csv")
print(" - step2_sample_strata_counts_main.csv")
print(" - step2_sample_strata_counts_steps.csv")
print(" - step2_sample_design_audit_sheet.csv")
print(" - step2_sample_summary.txt")

print("\nMain sample size:", len(main_sample))
print("Step sample size:", len(step_sample))

print("\nMain sample by style:")
if "style" in main_sample.columns:
    print(main_sample["style"].value_counts(dropna=False))

print("\nMain sample by Layer 2 observability:")
print(main_sample["layer2_observable"].value_counts(dropna=False))

print("\nMain sample by Base / Non-Base:")
print(main_sample["base_subset_flag"].value_counts(dropna=False))

if "complex_same_style_flag" in main_sample.columns:
    print("\nMain sample by complex same-style flag:")
    print(main_sample["complex_same_style_flag"].value_counts(dropna=False))

Reading MainDataset...
MainDataset shape: (8845, 130)
Reading step breakdown...
Step breakdown shape: (2859504, 51)

Saved outputs to: C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample
 - step2_sample_main_records.csv
 - step2_sample_step_records.csv
 - step2_sample_main_with_step_examples.csv
 - step2_sample_strata_counts_main.csv
 - step2_sample_strata_counts_steps.csv
 - step2_sample_design_audit_sheet.csv
 - step2_sample_summary.txt

Main sample size: 125
Step sample size: 35

Main sample by style:
style
Community      76
GMD            19
Third-Party    15
Custom         15
Name: count, dtype: int64

Main sample by Layer 2 observability:
layer2_observable
Yes    90
No     35
Name: count, dtype: int64

Main sample by Base / Non-Base:
base_subset_flag
Base        100
Non-Base     25
Name: count, dtype: int64

Main sample by complex same-style flag:
complex_same_style_flag
Yes    80
No     45
Name: count, dtype: int64
